# Speech-to-Text (STT) — local transcription on Apple Silicon

Pick a **model family** (Whisper, Whisper.en, Parakeet) and a **size** (SMALL / MEDIUM / LARGE), then run the
pipeline end to end:

1. Select the model from the `MODEL_NAME` registry (`APPLE → STT → FAMILY → SIZE`)
2. Review every family and size: parameters, architecture and memory
3. Load the model
4. Load an audio set (`audio/<set>.csv`: clips plus the text each one says)
5. Transcribe every clip, measuring speed, memory and energy
6. Save the transcripts
7. Score accuracy (word error rate) and the cost of the run
8. Compare SMALL / MEDIUM / LARGE of the selected family side by side

Whisper runs with `mlx-whisper` and Parakeet with `parakeet-mlx`, both on **MLX** on the Apple GPU.

## 1. Configuration — model registry

In [1]:
import os
from pathlib import Path

# APPLE -> STT -> family -> size -> model
MODEL_NAME = {
    'APPLE': {
        'STT': {
            'Whisper': {                      # OpenAI Whisper, multilingual
                'SMALL': 'whisper-small',
                'MEDIUM': 'whisper-medium',
                'LARGE': 'whisper-large-v3',
            },
            'Whisper.en': {                   # English-only Whisper variants
                'SMALL': 'whisper-base.en',
                'MEDIUM': 'whisper-small.en',
                'LARGE': 'whisper-medium.en',
            },
            'Parakeet': {                     # NVIDIA Parakeet (FastConformer encoder + TDT decoder), English
                'SMALL': 'parakeet-tdt_ctc-110m',
                'MEDIUM': 'parakeet-tdt-0.6b-v2',
                'LARGE': 'parakeet-tdt-1.1b',
            },
        },
    },
}

# Model -> (MLX checkpoint, library that runs it)
MLX_REPOS = {
    'whisper-small':         ('mlx-community/whisper-small-mlx',     'whisper'),
    'whisper-medium':        ('mlx-community/whisper-medium-mlx',    'whisper'),
    'whisper-large-v3':      ('mlx-community/whisper-large-v3-mlx',  'whisper'),
    'whisper-base.en':       ('mlx-community/whisper-base.en-mlx',   'whisper'),
    'whisper-small.en':      ('mlx-community/whisper-small.en-mlx',  'whisper'),
    'whisper-medium.en':     ('mlx-community/whisper-medium.en-mlx', 'whisper'),
    'parakeet-tdt_ctc-110m': ('mlx-community/parakeet-tdt_ctc-110m', 'parakeet'),
    'parakeet-tdt-0.6b-v2':  ('mlx-community/parakeet-tdt-0.6b-v2',  'parakeet'),
    'parakeet-tdt-1.1b':     ('mlx-community/parakeet-tdt-1.1b',     'parakeet'),
}

PLATFORM, TASK = 'APPLE', 'STT'
# Environment variables STT_FAMILY / STT_SIZE / STT_AUDIO / STT_COMPARE override these, for running from the terminal
FAMILY = os.environ.get('STT_FAMILY', 'Whisper')   # 'Whisper' | 'Whisper.en' | 'Parakeet'
SIZE = os.environ.get('STT_SIZE', 'MEDIUM')        # 'SMALL' | 'MEDIUM' | 'LARGE'
AUDIO_SET = os.environ.get('STT_AUDIO', 'stt_prompts_10')   # a CSV in audio/ with columns id,audio,voice,reference
RUN_FAMILY_COMPARISON = os.environ.get('STT_COMPARE', '1') == '1'   # section 8: also run the other two sizes

model_name = MODEL_NAME[PLATFORM][TASK][FAMILY][SIZE]
repo, backend = MLX_REPOS[model_name]

INPUT_PATH = Path('audio') / f'{AUDIO_SET}.csv'
OUTPUT_DIR = Path('outputs/stt') / FAMILY   # outputs/stt/<family>/<params>/<audio set>/ (see out_dir, section 2)
POWER_SAMPLE_MS = 100            # macmon sampling interval (its fastest is ~110 ms)
ELECTRICITY_USD_PER_KWH = 0.15   # your electricity price, for the $ per 1,000 audio hours figure
PARAKEET_CHUNK_S = 120           # Parakeet transcribes long audio in chunks of this length

print(f'{PLATFORM} / {TASK} / {FAMILY} / {SIZE} -> {model_name} -> {repo}  (library: {backend})')
print(f'Audio set: {INPUT_PATH}')

APPLE / STT / Whisper / SMALL -> whisper-small -> mlx-community/whisper-small-mlx  (library: whisper)
Audio set: audio/stt_prompts_10.csv


## 2. Model families — sizes, architecture and memory

Parameter counts are summed from each checkpoint's weights; layer counts come from its config.

- **Whisper** is an encoder-decoder transformer: the encoder turns 30-second windows of audio into features and
  the decoder writes the text token by token. Multilingual Whisper can also translate; the `.en` variants are
  trained on English only and are usually a little more accurate on English at the same size.
- **Parakeet** uses a FastConformer encoder with a TDT (token-and-duration transducer) decoder that can skip
  ahead several frames per step, which makes it much faster than Whisper. It is English only.
- These checkpoints are small enough to run at 16-bit, so no quantization is used: Whisper is stored in float16, and
  Parakeet is stored in float32 but loaded as bfloat16. Weights in memory ≈ params × 2 bytes.
- For Parakeet, the "decoder layers" are the LSTM layers of its small prediction network.

**What an input costs for a local model.** There is **no per-minute API fee ($0)**. What a minute of audio actually
costs is **time** (real-time factor), **memory** (peak GPU memory) and **energy** (measured in section 7 and
converted to $ at `ELECTRICITY_USD_PER_KWH`).

In [2]:
import mlx.core as mx
import pandas as pd

GB = 1e9
DEVICE = mx.device_info()

SPECS = pd.DataFrame([
    # model, params, encoder_layers, decoder_layers, width, mel_bins, languages, architecture, dtype, download_gb
    ('whisper-small', 240582932, 12, 12, 768, 80, 'multilingual', 'Transformer enc-dec', 'float16', 0.48),
    ('whisper-medium', 762321932, 24, 24, 1024, 80, 'multilingual', 'Transformer enc-dec', 'float16', 1.52),
    ('whisper-large-v3', 1541570580, 32, 32, 1280, 128, 'multilingual', 'Transformer enc-dec', 'float16', 3.08),
    ('whisper-base.en', 71825418, 6, 6, 512, 80, 'English', 'Transformer enc-dec', 'float16', 0.14),
    ('whisper-small.en', 240582182, 12, 12, 768, 80, 'English', 'Transformer enc-dec', 'float16', 0.48),
    ('whisper-medium.en', 762320932, 24, 24, 1024, 80, 'English', 'Transformer enc-dec', 'float16', 1.52),
    ('parakeet-tdt_ctc-110m', 114639495, 17, 1, 512, 80, 'English', 'FastConformer + TDT/CTC', 'bfloat16 (stored float32)', 0.46),
    ('parakeet-tdt-0.6b-v2', 617869958, 24, 2, 1024, 128, 'English', 'FastConformer + TDT', 'bfloat16 (stored float32)', 2.47),
    ('parakeet-tdt-1.1b', 1070516870, 42, 2, 1024, 80, 'English', 'FastConformer + TDT', 'bfloat16 (stored float32)', 4.28),
], columns=['model', 'params', 'encoder_layers', 'decoder_layers', 'width', 'mel_bins',
            'languages', 'architecture', 'dtype', 'download_gb']).set_index('model')

# Output subfolder per model, named by its parameter count: outputs/stt/<family>/<params>/<audio set>/
PARAM_FOLDER = {name: (f'{p / 1e9:.1f}B' if p >= 1e9 else f'{round(p / 1e6):d}M') for name, p in SPECS.params.items()}

def out_dir(name):
    d = OUTPUT_DIR / PARAM_FOLDER[name] / AUDIO_SET
    d.mkdir(parents=True, exist_ok=True)
    return d

rows = []
for family, sizes in MODEL_NAME[PLATFORM][TASK].items():
    for size, name in sizes.items():
        s = SPECS.loc[name]
        rows.append({
            '': '▶' if name == model_name else '',
            'Family': family,
            'Size': size,
            'Model': name,
            'Params (M)': f'{s.params / 1e6:,.0f}',
            'Architecture': s.architecture,
            'Encoder / decoder layers': f'{s.encoder_layers} / {s.decoder_layers}',
            'Width': s.width,
            'Mel bins': s.mel_bins,
            'Languages': s.languages,
            'Weights in memory (GB)': f'{s.params * 2 / GB:.2f}',
            'Download (GB)': f'{s.download_gb:.2f}',
            'API cost / audio minute': '$0 (local)',
        })

print(f"{DEVICE['device_name']}: {DEVICE['memory_size'] / GB:.0f} GB unified memory.  ▶ = selected model")
pd.DataFrame(rows).style.hide(axis='index')

Apple M5 Max: 52 GB unified memory.  ▶ = selected model


,Family,Size,Model,Params (M),Architecture,Encoder / decoder layers,Width,Mel bins,Languages,Weights in memory (GB),Download (GB),API cost / audio minute
▶,Whisper,SMALL,whisper-small,241,Transformer enc-dec,12 / 12,768,80,multilingual,0.48,0.48,$0 (local)
,Whisper,MEDIUM,whisper-medium,762,Transformer enc-dec,24 / 24,1024,80,multilingual,1.52,1.52,$0 (local)
,Whisper,LARGE,whisper-large-v3,"1,542",Transformer enc-dec,32 / 32,1280,128,multilingual,3.08,3.08,$0 (local)
,Whisper.en,SMALL,whisper-base.en,72,Transformer enc-dec,6 / 6,512,80,English,0.14,0.14,$0 (local)
,Whisper.en,MEDIUM,whisper-small.en,241,Transformer enc-dec,12 / 12,768,80,English,0.48,0.48,$0 (local)
,Whisper.en,LARGE,whisper-medium.en,762,Transformer enc-dec,24 / 24,1024,80,English,1.52,1.52,$0 (local)
,Parakeet,SMALL,parakeet-tdt_ctc-110m,115,FastConformer + TDT/CTC,17 / 1,512,80,English,0.23,0.46,$0 (local)
,Parakeet,MEDIUM,parakeet-tdt-0.6b-v2,618,FastConformer + TDT,24 / 2,1024,128,English,1.24,2.47,$0 (local)
,Parakeet,LARGE,parakeet-tdt-1.1b,"1,071",FastConformer + TDT,42 / 2,1024,80,English,2.14,4.28,$0 (local)


## 3. Load the model

In [3]:
import time
import mlx_whisper
from mlx_whisper.transcribe import ModelHolder
from parakeet_mlx import from_pretrained

def load_stt_model(name):
    """Returns (model, load seconds). Whisper models are cached in ModelHolder, which transcribe() reuses."""
    repo, backend = MLX_REPOS[name]
    t0 = time.perf_counter()
    if backend == 'whisper':
        ModelHolder.model, ModelHolder.model_path = None, None   # drop any previously cached Whisper model
        model = ModelHolder.get_model(repo, mx.float16)
    else:
        model = from_pretrained(repo)
    mx.eval(model.parameters())
    return model, time.perf_counter() - t0

model, load_seconds = load_stt_model(model_name)
spec = SPECS.loc[model_name]
print(f'Loaded {repo} in {load_seconds:.1f}s on {mx.default_device()}')
print(f'Parameters: {spec.params / 1e6:,.0f}M  |  {spec.architecture}, encoder {spec.encoder_layers} / decoder {spec.decoder_layers} layers')
print(f'Weights in memory: {mx.get_active_memory() / GB:.2f} GB ({spec['dtype']})')

/Users/genailab/mlx-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 4 files:   0%|                                                                                                                                                 | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 1393.23it/s]

Loaded mlx-community/whisper-small-mlx in 0.7s on Device(gpu, 0)
Parameters: 241M  |  Transformer enc-dec, encoder 12 / decoder 12 layers
Weights in memory: 0.48 GB (float16)


## 4. Load the audio set

`audio/<set>.csv` lists each clip with the exact text it says (`reference`), which section 7 uses to score
accuracy. The clips were generated with macOS `say` in several English accents, so the reference is exact.

In [4]:
import soundfile as sf

CLIPS = pd.read_csv(INPUT_PATH, dtype={'id': str})
CLIPS['id'] = CLIPS['id'].str.zfill(2)
CLIPS['duration_s'] = [sf.info(p).duration for p in CLIPS.audio]
CLIPS['reference_words'] = CLIPS.reference.str.split().str.len()

total_audio_s = CLIPS.duration_s.sum()
print(f'Audio set: {INPUT_PATH} ({len(CLIPS)} clips, {total_audio_s / 60:.1f} min of audio, '
      f'{CLIPS.reference_words.sum():,} reference words)')
pd.DataFrame({
    'id': CLIPS.id,
    'voice': CLIPS.voice,
    'duration (s)': CLIPS.duration_s.map('{:.1f}'.format),
    'words': CLIPS.reference_words,
    'reference': CLIPS.reference.map(lambda t: t if len(t) <= 80 else t[:77] + '...'),
}).style.hide(axis='index')

Audio set: audio/stt_prompts_10.csv (10 clips, 1.4 min of audio, 184 reference words)


id,voice,duration (s),words,reference
01,Samantha,6.9,18,Draft a professional email to a manager requesting a recommendation for a new...
02,Daniel,9.8,21,"Provide a brief background on the UK Prime Minister in 1995, highlighting key..."
03,Karen,6.9,19,"Explain the current phase of the moon, its significance, and how it fits with..."
04,Moira,8.9,19,"Describe the logic used to calculate the time for baking 96 cookies, consider..."
05,Rishi,19.1,41,"Imagine three friends, Mia, Noah, and Olivia, each running a race. Mia finish..."
06,Samantha,3.1,7,Tell me about Global warming.(In 500 words)
07,Daniel,3.3,7,Tell the History of USA.(In 500 words)
08,Karen,11.0,26,"Draft a compelling advertisement for a new grocery store in a small town, inc..."
09,Moira,7.3,13,"Explain the basic Blackjack game, including rules, logic, and example gamepla..."
10,Rishi,6.6,13,Breakdown the steering and breaking components of a Cessna 172N. (In 500 words)


## 5. Transcribe

Each clip is transcribed separately. Whisper runs greedy decoding in English with
`condition_on_previous_text=False`, which stops long audio from drifting into invented text. Parakeet splits audio
longer than `PARAKEET_CHUNK_S` into overlapping chunks.

**Power monitoring.** `macmon` (no sudo needed) runs in the background for the whole notebook, sampling the chip's
power sensors about every 110 ms. Each clip's energy is the power integrated over the time it took to transcribe.
If `macmon` isn't installed, the power metrics are left empty and everything else runs. Readings are Apple's
on-chip estimates, not a wall-meter measurement, and very short clips are covered by only a few samples.

In [5]:
import atexit
import json
import shutil
import subprocess
import threading

class PowerMonitor:
    """Runs `macmon pipe` in the background and keeps every sample.

    Each sample is the average over the interval since the previous one, so the energy in any time
    window is the sum of power x (overlap of that interval with the window).
    """
    def __init__(self, interval_ms):
        self.samples = []  # (time.perf_counter(), macmon sample dict)
        self.available = shutil.which('macmon') is not None
        if not self.available:
            print('macmon not found: power and energy metrics will be empty (install: brew install macmon)')
            return
        self.proc = subprocess.Popen(['macmon', 'pipe', '-i', str(interval_ms)],
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        atexit.register(self.proc.terminate)
        threading.Thread(target=self._read, daemon=True).start()
        t_start = time.perf_counter()
        while len(self.samples) < 2 and time.perf_counter() - t_start < 5:  # macmon takes ~0.7 s to start
            time.sleep(0.05)
        self.available = len(self.samples) >= 2

    def _read(self):
        for line in self.proc.stdout:
            try:
                self.samples.append((time.perf_counter(), json.loads(line)))
            except json.JSONDecodeError:
                pass

    def wait_past(self, t, timeout=2.0):
        """Block until a sample covering time t has arrived."""
        deadline = time.perf_counter() + timeout
        while self.samples[-1][0] < t and time.perf_counter() < deadline:
            time.sleep(0.02)

    def window(self, a, b):
        """Samples overlapping [a, b], as (seconds of overlap, sample)."""
        return [(min(t, b) - max(t_prev, a), s)
                for (t_prev, _), (t, s) in zip(self.samples, self.samples[1:])
                if min(t, b) - max(t_prev, a) > 0]

def soc_power(s):
    return s['all_power'] + s['ram_power']  # CPU + GPU + Neural Engine + memory, in watts

def integrate(win, fn):
    return sum(overlap * fn(s) for overlap, s in win)

POWER_KEYS = ['gpu_util_avg_pct', 'gpu_power_avg_w', 'gpu_power_peak_w', 'soc_power_avg_w', 'gpu_temp_max_c',
              'energy_j', 'energy_net_j', 'power_samples']

def summarize_power(t0, t_end):
    if not POWER.available:
        return dict.fromkeys(POWER_KEYS)
    POWER.wait_past(t_end)
    run = POWER.window(t0, t_end)
    samples = [s for _, s in run]
    dur = t_end - t0
    e_soc = integrate(run, soc_power)
    return {
        'gpu_util_avg_pct': round(100 * integrate(run, lambda s: s['gpu_active_ratio']) / dur, 1),
        'gpu_power_avg_w': round(integrate(run, lambda s: s['gpu_power']) / dur, 1),
        'gpu_power_peak_w': round(max(s['gpu_power'] for s in samples), 1),
        'soc_power_avg_w': round(e_soc / dur, 1),
        'gpu_temp_max_c': round(max(s['temp']['gpu_temp_avg'] for s in samples), 1),
        'energy_j': round(e_soc, 2),
        'energy_net_j': round(e_soc - IDLE_SOC_W * dur, 2),
        'power_samples': len(run),
    }

POWER = PowerMonitor(POWER_SAMPLE_MS)
if POWER.available:
    time.sleep(1.0)  # idle baseline, measured with the model loaded but not running
    a, b = POWER.samples[0][0], POWER.samples[-1][0]
    IDLE_SOC_W = integrate(POWER.window(a, b), soc_power) / (b - a)
    print(f'Power monitor running (macmon, ~{POWER_SAMPLE_MS} ms samples). Idle chip power: {IDLE_SOC_W:.2f} W')

Power monitor running (macmon, ~100 ms samples). Idle chip power: 0.28 W


In [6]:
def transcribe(model, name, audio_path, duration_s):
    repo, backend = MLX_REPOS[name]
    mx.reset_peak_memory()
    t0 = time.perf_counter()
    if backend == 'whisper':
        language = None if name.endswith('.en') else 'en'
        text = mlx_whisper.transcribe(audio_path, path_or_hf_repo=repo, language=language, verbose=None,
                                      condition_on_previous_text=False, fp16=True)['text']
    else:
        chunk = PARAKEET_CHUNK_S if duration_s > PARAKEET_CHUNK_S else None
        text = model.transcribe(audio_path, chunk_duration=chunk, overlap_duration=15.0).text
    t_end = time.perf_counter()
    return {
        'text': text.strip(),
        'time_s': t_end - t0,
        'peak_gb': mx.get_peak_memory() / GB,
        'power': summarize_power(t0, t_end),
    }

def warm_up(model, name):
    """Transcribe the first clip once, untimed, so GPU kernels are compiled before anything is measured."""
    transcribe(model, name, CLIPS.audio.iloc[0], CLIPS.duration_s.iloc[0])

def run_audio_set(model, name):
    warm_up(model, name)
    runs = []
    for row in CLIPS.itertuples(index=False):
        t = transcribe(model, name, row.audio, row.duration_s)
        t.update(id=row.id, voice=row.voice, audio_s=row.duration_s, reference=row.reference)
        runs.append(t)
        print(f"  [{row.id}] {row.duration_s:6.1f}s audio -> {t['time_s']:5.2f}s  "
              f"({row.duration_s / t['time_s']:6.1f}x real time, {row.voice})")
    return runs

print(f'Transcribing {len(CLIPS)} clips with {model_name}...')
runs = run_audio_set(model, model_name)
print(f"Done: {total_audio_s:.0f}s of audio in {sum(r['time_s'] for r in runs):.1f}s")

Transcribing 10 clips with whisper-small...


  [01]    6.9s audio ->  0.11s  (  63.2x real time, Samantha)


  [02]    9.8s audio ->  0.12s  (  84.8x real time, Daniel)


  [03]    6.9s audio ->  0.11s  (  64.1x real time, Karen)


  [04]    8.9s audio ->  0.11s  (  80.2x real time, Moira)


  [05]   19.1s audio ->  0.19s  ( 100.7x real time, Rishi)
  [06]    3.1s audio ->  0.07s  (  41.2x real time, Samantha)


  [07]    3.3s audio ->  0.07s  (  44.7x real time, Daniel)


  [08]   11.0s audio ->  0.13s  (  86.1x real time, Karen)


  [09]    7.3s audio ->  0.10s  (  72.0x real time, Moira)
  [10]    6.6s audio ->  0.10s  (  67.5x real time, Rishi)
Done: 83s of audio in 1.1s


## 6. Save the transcripts

One text file per clip: `outputs/stt/<family>/<params>/<audio set>/<model>_transcript_<id>.txt`.

In [7]:
import textwrap

def save_transcripts(name, runs):
    d = out_dir(name)
    for r in runs:
        (d / f"{name}_transcript_{r['id']}.txt").write_text(r['text'] + '\n')
    return d

saved_to = save_transcripts(model_name, runs)
print(f'Saved {len(runs)} transcripts to {saved_to}/\n')
first = runs[0]
print(f"=== Clip {first['id']} ({first['voice']}) ===")
print('Reference: ' + textwrap.shorten(first['reference'], 300))
print('Transcript: ' + textwrap.shorten(first['text'], 300))

Saved 10 transcripts to outputs/stt/Whisper/241M/stt_prompts_10/

=== Clip 01 (Samantha) ===
Reference: Draft a professional email to a manager requesting a recommendation for a new job opportunity. (In 500 words)
Transcript: Draft a professional email to a manager requesting a recommendation for a new job opportunity, in 500 words.


## 7. Accuracy and cost of this run

**Word error rate (WER)** = (substituted + inserted + deleted words) ÷ reference words, after normalising both texts
(lower case, punctuation removed, number words turned into digits, so "five hundred" and "500" match). The set's
WER is pooled over all its words. **Real-time factor** = processing time ÷ audio length (lower is faster).

In [8]:
import re
import difflib

UNITS = {w: i for i, w in enumerate(
    'zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen '
    'fifteen sixteen seventeen eighteen nineteen'.split())}
TENS = {w: 10 * i for i, w in enumerate('twenty thirty forty fifty sixty seventy eighty ninety'.split(), 2)}

def words_to_digits(words):
    out, num, active = [], 0, False
    for w in words:
        if w in UNITS:
            num += UNITS[w]; active = True
        elif w in TENS:
            num += TENS[w]; active = True
        elif w == 'hundred' and active:
            num *= 100
        elif w == 'thousand' and active:
            num *= 1000
        elif w == 'and' and active:
            continue
        else:
            if active:
                out.append(str(num)); num, active = 0, False
            out.append(w)
    if active:
        out.append(str(num))
    return out

def normalize(text):
    text = text.lower().replace('’', "'").replace('-', ' ')
    text = re.sub(r'\.(?!\d)', '', text)                        # U.K. -> uk, keep decimals
    text = re.sub(r"[^a-z0-9' ]+", ' ', text)
    text = re.sub(r'(\d)([a-z])\b', r'\1 \2', text)             # 172n -> 172 n (matches "172 N")
    return words_to_digits(text.split())

def edit_counts(ref, hyp):
    """Substitutions, deletions, insertions between two word lists."""
    s = d = i = 0
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes():
        if op == 'replace':
            s += min(a1 - a0, b1 - b0); d += max(0, (a1 - a0) - (b1 - b0)); i += max(0, (b1 - b0) - (a1 - a0))
        elif op == 'delete':
            d += a1 - a0
        elif op == 'insert':
            i += b1 - b0
    return s, d, i

def set_metrics(name, runs, load_s):
    s = SPECS.loc[name]
    per = []
    for r in runs:
        ref, hyp = normalize(r['reference']), normalize(r['text'])
        sub, dele, ins = edit_counts(ref, hyp)
        per.append({
            'id': r['id'], 'voice': r['voice'], 'audio_s': round(r['audio_s'], 2), 'time_s': round(r['time_s'], 3),
            'real_time_factor': round(r['time_s'] / r['audio_s'], 4), 'x_real_time': round(r['audio_s'] / r['time_s'], 1),
            'reference_words': len(ref), 'substitutions': sub, 'deletions': dele, 'insertions': ins,
            'wer_pct': round(100 * (sub + dele + ins) / len(ref), 2), 'peak_memory_gb': round(r['peak_gb'], 2),
            **r['power'],
        })
    audio_s = sum(p['audio_s'] for p in per)
    time_s = sum(p['time_s'] for p in per)
    ref_words = sum(p['reference_words'] for p in per)
    errors = sum(p['substitutions'] + p['deletions'] + p['insertions'] for p in per)
    measured = per[0]['energy_j'] is not None
    energy = sum(p['energy_j'] for p in per) if measured else None
    weighted = lambda k: round(sum(p[k] * p['time_s'] for p in per) / time_s, 1) if measured else None
    j_per_min = energy / (audio_s / 60) if measured else None
    summary = {
        'family': next(f for f, sz in MODEL_NAME[PLATFORM][TASK].items() if name in sz.values()),
        'model': name, 'repo': MLX_REPOS[name][0], 'library': MLX_REPOS[name][1],
        'params_m': round(s.params / 1e6), 'audio_set': AUDIO_SET, 'clips': len(per),
        'load_s': round(load_s, 1), 'audio_s': round(audio_s, 1), 'time_s': round(time_s, 2),
        'real_time_factor': round(time_s / audio_s, 4), 'x_real_time': round(audio_s / time_s, 1),
        'reference_words': ref_words, 'word_errors': errors, 'wer_pct': round(100 * errors / ref_words, 2),
        'peak_memory_gb': max(p['peak_memory_gb'] for p in per), 'api_cost_usd': 0.0,
        'gpu_util_avg_pct': weighted('gpu_util_avg_pct'), 'gpu_power_avg_w': weighted('gpu_power_avg_w'),
        'gpu_power_peak_w': max(p['gpu_power_peak_w'] for p in per) if measured else None,
        'soc_power_avg_w': weighted('soc_power_avg_w'),
        'gpu_temp_max_c': max(p['gpu_temp_max_c'] for p in per) if measured else None,
        'energy_j': round(energy, 1) if measured else None,
        'energy_j_per_audio_minute': round(j_per_min, 2) if measured else None,
        # J/min x 60,000 min = energy for 1,000 audio hours; 1 kWh = 3.6e6 J
        'usd_per_1000_audio_hours': round(j_per_min * 60_000 / 3.6e6 * ELECTRICITY_USD_PER_KWH, 4) if measured else None,
        'electricity_usd_per_kwh': ELECTRICITY_USD_PER_KWH,
    }
    return {'summary': summary, 'per_clip': per}

result = set_metrics(model_name, runs, load_seconds)
sm = result['summary']

rows = [
    ('Clips / audio', f"{sm['clips']} / {sm['audio_s'] / 60:.1f} min"),
    ('Word error rate (pooled)', f"{sm['wer_pct']}% ({sm['word_errors']} errors in {sm['reference_words']:,} words)"),
    ('Processing time', f"{sm['time_s']} s"),
    ('Speed', f"{sm['x_real_time']}x real time (real-time factor {sm['real_time_factor']})"),
    ('Peak memory', f"{sm['peak_memory_gb']} GB"),
    ('API cost', '$0.00 (runs locally)'),
]
if sm['energy_j'] is not None:
    rows += [
        ('GPU utilisation (avg) / GPU power (avg, peak)', f"{sm['gpu_util_avg_pct']}% / {sm['gpu_power_avg_w']} W, {sm['gpu_power_peak_w']} W"),
        ('Chip power, avg (CPU + GPU + memory)', f"{sm['soc_power_avg_w']} W"),
        ('GPU temperature (max)', f"{sm['gpu_temp_max_c']} °C"),
        ('Energy for the whole set', f"{sm['energy_j']} J"),
        ('Energy per minute of audio', f"{sm['energy_j_per_audio_minute']} J"),
        (f'Electricity per 1,000 hours of audio (at ${ELECTRICITY_USD_PER_KWH}/kWh)', f"${sm['usd_per_1000_audio_hours']}"),
    ]
display(pd.DataFrame(rows, columns=['Metric', model_name]).style.hide(axis='index'))

pd.DataFrame([{
    'id': p['id'], 'voice': p['voice'], 'audio (s)': f"{p['audio_s']:.1f}", 'time (s)': f"{p['time_s']:.2f}",
    'x real time': f"{p['x_real_time']:.1f}", 'WER %': f"{p['wer_pct']:.1f}",
    'sub / del / ins': f"{p['substitutions']} / {p['deletions']} / {p['insertions']}",
    'energy (J)': '' if p['energy_j'] is None else f"{p['energy_j']:.1f}",
} for p in result['per_clip']]).style.hide(axis='index')

Metric,whisper-small
Clips / audio,10 / 1.4 min
Word error rate (pooled),6.22% (12 errors in 193 words)
Processing time,1.11 s
Speed,74.7x real time (real-time factor 0.0134)
Peak memory,1.45 GB
API cost,$0.00 (runs locally)
"GPU utilisation (avg) / GPU power (avg, peak)","44.4% / 6.2 W, 8.0 W"
"Chip power, avg (CPU + GPU + memory)",11.4 W
GPU temperature (max),36.7 °C
Energy for the whole set,12.6 J


id,voice,audio (s),time (s),x real time,WER %,sub / del / ins,energy (J)
01,Samantha,6.9,0.11,63.2,0.0,0 / 0 / 0,0.8
02,Daniel,9.8,0.12,84.8,4.5,1 / 0 / 0,1.2
03,Karen,6.9,0.11,64.1,0.0,0 / 0 / 0,1.3
04,Moira,8.9,0.11,80.2,0.0,0 / 0 / 0,1.0
05,Rishi,19.1,0.19,100.7,9.5,4 / 0 / 0,2.9
06,Samantha,3.1,0.07,41.2,0.0,0 / 0 / 0,1.1
07,Daniel,3.3,0.07,44.7,0.0,0 / 0 / 0,0.8
08,Karen,11.0,0.13,86.1,0.0,0 / 0 / 0,1.8
09,Moira,7.3,0.10,72.0,14.3,1 / 0 / 1,0.9
10,Rishi,6.6,0.10,67.5,35.7,3 / 1 / 1,0.8


In [9]:
# Every word the model got wrong, clip by clip (after normalisation)
for r in runs:
    ref, hyp = normalize(r['reference']), normalize(r['text'])
    diffs = [(op, ' '.join(ref[a0:a1]), ' '.join(hyp[b0:b1]))
             for op, a0, a1, b0, b1 in difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes() if op != 'equal']
    if diffs:
        print(f"[{r['id']}] " + ';  '.join(f'{a!r} -> {b!r}' for _, a, b in diffs[:8]) + (' ...' if len(diffs) > 8 else ''))

[02] 'prime' -> 'crime'
[05] 'noah' -> 'noa';  'noah' -> 'noa';  'noah' -> 'noa';  "mia's" -> "maya's"
[09] 'gameplay' -> 'game play'
[10] 'breakdown' -> 'break down';  'breaking' -> 'braking';  'a cessna' -> 'asesna'


### Simple metrics

The metrics saved for each model, as `<model>_simple_metrics.json` in its output folder: plain names with the unit in every key, grouped into audio, accuracy, speed, memory, GPU,
energy and cost, plus one short entry per clip.

In [10]:
def simple_metrics(name, result):
    """A flat, readable summary of one model's run: plain names, units in every key."""
    sm, per = result['summary'], result['per_clip']
    s = SPECS.loc[name]
    size = next(sz for sizes in MODEL_NAME[PLATFORM][TASK].values() for sz, n in sizes.items() if n == name)
    n = len(per)
    durations = [p['audio_s'] for p in per]
    worst = max(per, key=lambda p: p['wer_pct'])
    energy = sm['energy_j']
    measured = energy is not None
    price = sm['electricity_usd_per_kwh']
    return {
        'model': name,
        'family': sm['family'],
        'size': size,
        'repo': sm['repo'],
        'library': sm['library'],
        'parameters_million': sm['params_m'],
        'architecture': s['architecture'],
        'precision': s['dtype'],
        'languages': s['languages'],
        'audio_set': sm['audio_set'],
        'clips_run': n,
        'audio': {
            'total_seconds': sm['audio_s'],
            'total_minutes': round(sm['audio_s'] / 60, 2),
            'avg_clip_seconds': round(sum(durations) / n, 1),
            'shortest_clip_seconds': min(durations),
            'longest_clip_seconds': max(durations),
            'reference_words': sm['reference_words'],
        },
        'accuracy': {
            'word_error_rate_pct': sm['wer_pct'],
            'word_accuracy_pct': round(100 - sm['wer_pct'], 2),
            'word_errors': sm['word_errors'],
            'substitutions': sum(p['substitutions'] for p in per),
            'deletions': sum(p['deletions'] for p in per),
            'insertions': sum(p['insertions'] for p in per),
            'perfect_clips': sum(1 for p in per if p['wer_pct'] == 0),
            'worst_clip_id': worst['id'],
            'worst_clip_word_error_rate_pct': worst['wer_pct'],
        },
        'speed': {
            'model_load_seconds': sm['load_s'],
            'total_processing_seconds': sm['time_s'],
            'avg_seconds_per_clip': round(sm['time_s'] / n, 3),
            'times_faster_than_real_time': sm['x_real_time'],
            'real_time_factor': sm['real_time_factor'],
            'minutes_to_transcribe_1_hour_of_audio': round(60 * sm['real_time_factor'], 2),
        },
        'memory': {
            'weights_gb': round(s.params * 2 / 1e9, 2),
            'peak_memory_gb': sm['peak_memory_gb'],
        },
        'gpu': {
            'avg_utilization_pct': sm['gpu_util_avg_pct'],
            'avg_power_watts': sm['gpu_power_avg_w'],
            'peak_power_watts': sm['gpu_power_peak_w'],
            'max_temperature_c': sm['gpu_temp_max_c'],
            'whole_chip_avg_power_watts': sm['soc_power_avg_w'],
        },
        'energy': {
            'total_joules': energy,
            'total_watt_hours': round(energy / 3600, 4) if measured else None,
            'avg_joules_per_clip': round(energy / n, 2) if measured else None,
            'per_audio_minute_joules': sm['energy_j_per_audio_minute'],
            'per_audio_hour_watt_hours': round(sm['energy_j_per_audio_minute'] * 60 / 3600, 3) if measured else None,
        },
        'cost_usd': {
            'api_fee': 0.0,
            'electricity_whole_run': round(energy / 3.6e6 * price, 7) if measured else None,
            'electricity_per_1000_audio_hours': sm['usd_per_1000_audio_hours'],
            'electricity_price_per_kwh': price,
        },
        'per_clip': [{
            'id': p['id'],
            'voice': p['voice'],
            'audio_seconds': p['audio_s'],
            'processing_seconds': p['time_s'],
            'times_faster_than_real_time': p['x_real_time'],
            'word_error_rate_pct': p['wer_pct'],
            'word_errors': p['substitutions'] + p['deletions'] + p['insertions'],
            'energy_joules': p['energy_j'],
        } for p in per],
    }

def save_simple(name, result, folder):
    path = Path(folder) / f'{name}_simple_metrics.json'
    path.write_text(json.dumps(simple_metrics(name, result), indent=2))
    return path

# --- write it for the model run above
print(f'Saved {save_simple(model_name, result, out_dir(model_name))}')
print(json.dumps({k: v for k, v in simple_metrics(model_name, result).items() if k != 'per_clip'}, indent=2))

Saved outputs/stt/Whisper/241M/stt_prompts_10/whisper-small_simple_metrics.json
{
  "model": "whisper-small",
  "family": "Whisper",
  "size": "SMALL",
  "repo": "mlx-community/whisper-small-mlx",
  "library": "whisper",
  "parameters_million": 241,
  "architecture": "Transformer enc-dec",
  "precision": "float16",
  "languages": "multilingual",
  "audio_set": "stt_prompts_10",
  "clips_run": 10,
  "audio": {
    "total_seconds": 82.8,
    "total_minutes": 1.38,
    "avg_clip_seconds": 8.3,
    "shortest_clip_seconds": 3.08,
    "longest_clip_seconds": 19.13,
    "reference_words": 193
  },
  "accuracy": {
    "word_error_rate_pct": 6.22,
    "word_accuracy_pct": 93.78,
    "word_errors": 12,
    "substitutions": 9,
    "deletions": 1,
    "insertions": 2,
    "perfect_clips": 6,
    "worst_clip_id": "10",
    "worst_clip_word_error_rate_pct": 35.71
  },
  "speed": {
    "model_load_seconds": 0.7,
    "total_processing_seconds": 1.11,
    "avg_seconds_per_clip": 0.111,
    "times_faste

## 8. Compare every size in the family

Runs the whole audio set through SMALL, MEDIUM and LARGE of `FAMILY` (reusing the results already produced for
`SIZE`), unloading each model before the next. Set `STT_COMPARE=0` (or `RUN_FAMILY_COMPARISON = False` in
section 1) to skip.

In [11]:
import gc

results = {SIZE: (result, runs)}
if RUN_FAMILY_COMPARISON:
    del model
    gc.collect()
    mx.clear_cache()
    for size, name in MODEL_NAME[PLATFORM][TASK][FAMILY].items():
        if size == SIZE:
            continue
        print(f'{size}: {name}')
        m, load_s = load_stt_model(name)
        r = run_audio_set(m, name)
        save_transcripts(name, r)
        res = set_metrics(name, r, load_s)
        save_simple(name, res, out_dir(name))
        results[size] = (res, r)
        del m
        gc.collect()
        mx.clear_cache()

order = [s for s in ('SMALL', 'MEDIUM', 'LARGE') if s in results]
compare = pd.DataFrame({
    s: {
        'Model': sm_['model'],
        'Params (M)': f"{sm_['params_m']:,}",
        'Load time (s)': sm_['load_s'],
        'WER (%)': sm_['wer_pct'],
        'Word errors': sm_['word_errors'],
        'Processing time (s)': sm_['time_s'],
        'Speed (x real time)': sm_['x_real_time'],
        'Peak memory (GB)': sm_['peak_memory_gb'],
        'GPU util avg (%)': sm_['gpu_util_avg_pct'],
        'GPU power avg (W)': sm_['gpu_power_avg_w'],
        'Energy, whole set (J)': sm_['energy_j'],
        'Energy / audio minute (J)': sm_['energy_j_per_audio_minute'],
        '$ / 1,000 audio hours (electricity)': sm_['usd_per_1000_audio_hours'],
    } for s in order for sm_ in [results[s][0]['summary']]
})
compare.columns.name = FAMILY
compare

Whisper,SMALL
Model,whisper-small
Params (M),241
Load time (s),0.7
WER (%),6.22
Word errors,12
Processing time (s),1.11
Speed (x real time),74.7
Peak memory (GB),1.45
GPU util avg (%),44.4
GPU power avg (W),6.2


In [12]:
# WER (%) per clip and size
pd.DataFrame({s: {p['id']: p['wer_pct'] for p in results[s][0]['per_clip']} for s in order}).rename_axis('clip id')

,SMALL
clip id,
01,0.00
02,4.55
03,0.00
04,0.00
05,9.52
06,0.00
07,0.00
08,0.00
09,14.29


## 9. All models on this audio set

Collects every `<model>_simple_metrics.json` produced so far for `AUDIO_SET` (any family, any size, from this
notebook or terminal runs) into `outputs/stt/all_models_<audio set>.json` and `.csv`.

In [13]:
FAMILY_ORDER = list(MODEL_NAME[PLATFORM][TASK])
SIZE_ORDER = ['SMALL', 'MEDIUM', 'LARGE']
files = Path('outputs/stt').glob(f'*/*/{AUDIO_SET}/*_simple_metrics.json')
models = sorted((json.loads(f.read_text()) for f in files),
                key=lambda m: (FAMILY_ORDER.index(m['family']), SIZE_ORDER.index(m['size'])))
overview = [{k: v for k, v in m.items() if k != 'per_clip'} for m in models]
all_json = Path('outputs/stt') / f'all_models_{AUDIO_SET}.json'
all_csv = all_json.with_suffix('.csv')
all_json.write_text(json.dumps({'audio_set': AUDIO_SET, 'models': overview}, indent=2))
flat = pd.json_normalize(overview)
flat.to_csv(all_csv, index=False)
print(f'{len(models)} models -> {all_json} and {all_csv}')

pd.DataFrame({
    'Family': flat['family'], 'Size': flat['size'], 'Model': flat['model'],
    'Params (M)': flat['parameters_million'],
    'WER (%)': flat['accuracy.word_error_rate_pct'],
    'Perfect clips': flat['accuracy.perfect_clips'],
    'x real time': flat['speed.times_faster_than_real_time'],
    'Min per audio hour': flat['speed.minutes_to_transcribe_1_hour_of_audio'],
    'Peak memory (GB)': flat['memory.peak_memory_gb'],
    'GPU power (W)': flat['gpu.avg_power_watts'],
    'Energy (J)': flat['energy.total_joules'],
    'J / audio minute': flat['energy.per_audio_minute_joules'],
    '$ / 1,000 audio hours': flat['cost_usd.electricity_per_1000_audio_hours'],
}).style.hide(axis='index')

9 models -> outputs/stt/all_models_stt_prompts_10.json and outputs/stt/all_models_stt_prompts_10.csv


Family,Size,Model,Params (M),WER (%),Perfect clips,x real time,Min per audio hour,Peak memory (GB),GPU power (W),Energy (J),J / audio minute,"$ / 1,000 audio hours"
Whisper,SMALL,whisper-small,241,6.220000,6,74.700000,0.800000,1.450000,6.200000,12.600000,9.120000,0.022800
Whisper,MEDIUM,whisper-medium,762,2.070000,8,42.300000,1.420000,2.220000,16.000000,51.000000,36.920000,0.092300
Whisper,LARGE,whisper-large-v3,1542,3.110000,7,27.200000,2.210000,4.020000,22.400000,104.600000,75.790000,0.189500
Whisper.en,SMALL,whisper-base.en,72,2.070000,8,122.500000,0.490000,0.550000,2.400000,5.700000,4.160000,0.010400
Whisper.en,MEDIUM,whisper-small.en,241,2.070000,8,77.400000,0.770000,1.450000,6.900000,13.200000,9.600000,0.024000
Whisper.en,LARGE,whisper-medium.en,762,3.630000,7,43.100000,1.390000,2.220000,16.600000,52.400000,37.970000,0.094900
Parakeet,SMALL,parakeet-tdt_ctc-110m,115,5.180000,7,93.800000,0.640000,0.630000,1.800000,6.700000,4.820000,0.012100
Parakeet,MEDIUM,parakeet-tdt-0.6b-v2,618,3.110000,8,83.000000,0.730000,2.390000,5.000000,11.300000,8.210000,0.020500
Parakeet,LARGE,parakeet-tdt-1.1b,1071,4.660000,7,79.600000,0.760000,3.150000,7.900000,14.500000,10.480000,0.026200
